In [1]:
import os 
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import json 
import random 


In [2]:
train_path = "100k_aya_expanse_gliner.json"
# train_path = "../data.json"

with open(train_path,"r") as f:
    data = json.load(f)

print('Dataset size:', len(data))

random.shuffle(data)
print('Dataset is shuffled...')

train_dataset = data[:int(len(data)*0.9)]
test_dataset = data[int(len(data)*0.9):]

print('Dataset is splitted...')


Dataset size: 86116
Dataset is shuffled...
Dataset is splitted...


In [3]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "true"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import torch
torch.cuda.set_device('cuda:0')
from gliner import GLiNERConfig, GLiNER
from gliner.training import Trainer, TrainingArguments
from gliner.data_processing.collator import DataCollatorWithPadding, DataCollator
from gliner.utils import load_config_as_namespace
from gliner.data_processing import WordsSplitter, GLiNERDataset

In [4]:
model_name = "NAMAA-Space/gliner_arabert_base_v02"

In [5]:
config_gliner = GLiNERConfig(model_name)
config_gliner


GLiNERConfig {
  "class_token_index": -1,
  "dropout": 0.4,
  "embed_ent_token": true,
  "encoder_config": null,
  "ent_token": "<<ENT>>",
  "fine_tune": true,
  "fuse_layers": false,
  "has_rnn": true,
  "hidden_size": 512,
  "labels_encoder": null,
  "labels_encoder_config": null,
  "max_len": 384,
  "max_neg_type_ratio": 1,
  "max_types": 25,
  "max_width": 12,
  "model_name": "NAMAA-Space/gliner_arabert_base_v02",
  "model_type": "gliner",
  "name": "span level gliner",
  "num_post_fusion_layers": 1,
  "post_fusion_schema": "",
  "sep_token": "<<SEP>>",
  "span_mode": "markerV0",
  "subtoken_pooling": "first",
  "transformers_version": "4.49.0",
  "vocab_size": -1,
  "words_splitter_type": "whitespace"
}

In [6]:
model = GLiNER(config_gliner)
model

Some weights of BertModel were not initialized from the model checkpoint at NAMAA-Space/gliner_arabert_base_v02 and are newly initialized: ['embeddings.LayerNorm.bias', 'embeddings.LayerNorm.weight', 'embeddings.position_embeddings.weight', 'embeddings.token_type_embeddings.weight', 'embeddings.word_embeddings.weight', 'encoder.layer.0.attention.output.LayerNorm.bias', 'encoder.layer.0.attention.output.LayerNorm.weight', 'encoder.layer.0.attention.output.dense.bias', 'encoder.layer.0.attention.output.dense.weight', 'encoder.layer.0.attention.self.key.bias', 'encoder.layer.0.attention.self.key.weight', 'encoder.layer.0.attention.self.query.bias', 'encoder.layer.0.attention.self.query.weight', 'encoder.layer.0.attention.self.value.bias', 'encoder.layer.0.attention.self.value.weight', 'encoder.layer.0.intermediate.dense.bias', 'encoder.layer.0.intermediate.dense.weight', 'encoder.layer.0.output.LayerNorm.bias', 'encoder.layer.0.output.LayerNorm.weight', 'encoder.layer.0.output.dense.bias'

GLiNER(
  (model): SpanModel(
    (token_rep_layer): Encoder(
      (bert_layer): Transformer(
        (model): BertModel(
          (embeddings): BertEmbeddings(
            (word_embeddings): Embedding(30522, 768, padding_idx=0)
            (position_embeddings): Embedding(512, 768)
            (token_type_embeddings): Embedding(2, 768)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (encoder): BertEncoder(
            (layer): ModuleList(
              (0-11): 12 x BertLayer(
                (attention): BertAttention(
                  (self): BertSdpaSelfAttention(
                    (query): Linear(in_features=768, out_features=768, bias=True)
                    (key): Linear(in_features=768, out_features=768, bias=True)
                    (value): Linear(in_features=768, out_features=768, bias=True)
                    (dropout): Dropout(p=0.1, inplace=False)
            

In [7]:
device = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')
device


device(type='cuda', index=0)

In [8]:
model

GLiNER(
  (model): SpanModel(
    (token_rep_layer): Encoder(
      (bert_layer): Transformer(
        (model): BertModel(
          (embeddings): BertEmbeddings(
            (word_embeddings): Embedding(30522, 768, padding_idx=0)
            (position_embeddings): Embedding(512, 768)
            (token_type_embeddings): Embedding(2, 768)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (encoder): BertEncoder(
            (layer): ModuleList(
              (0-11): 12 x BertLayer(
                (attention): BertAttention(
                  (self): BertSdpaSelfAttention(
                    (query): Linear(in_features=768, out_features=768, bias=True)
                    (key): Linear(in_features=768, out_features=768, bias=True)
                    (value): Linear(in_features=768, out_features=768, bias=True)
                    (dropout): Dropout(p=0.1, inplace=False)
            

In [9]:
# Count the total number of parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")

# Optionally, count trainable parameters only
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable_params:,}")

Total number of parameters: 120,900,352
Trainable parameters: 120,900,352


In [10]:
model.to(device)
gpu_memory = torch.cuda.memory_allocated() / (1024 ** 2)  # In MB
print(f"GPU memory used: {gpu_memory:.2f} MB")

GPU memory used: 462.79 MB


In [11]:
data_collator = DataCollator(model.config, data_processor=model.data_processor, prepare_labels=True)


In [12]:
model.to(device)
print("done")

done


In [13]:

# import torch
# torch.cuda.empty_cache()
print("Available GPUs:", torch.cuda.device_count())
print("Current device:", torch.cuda.current_device())
# print("Trainer args:", training_args.device)

Available GPUs: 1
Current device: 0


In [14]:
data_collator = DataCollator(model.config, data_processor=model.data_processor, prepare_labels=True)


In [15]:
train_dataset

[{'tokenized_text': ['مقدمة',
   'ابن',
   'خلدون',
   'هي',
   'كتاب',
   'يتناول',
   'التاريخ',
   '،',
   'كتبها',
   'ستيفن',
   'كوفي',
   'ونشرها',
   'مركز',
   'الأدب',
   'العربي',
   '.',
   'يركز',
   'الكتاب',
   'على',
   'التفاؤل',
   'ودوره',
   'في',
   'نجاح',
   'الأفراد',
   'والمنظمات',
   '.'],
  'ner': [[0, 2, 'اسم الكتاب'],
   [9, 10, 'المؤلف'],
   [12, 14, 'دار النشر'],
   [19, 19, 'الموضوع الرئيسي']]},
 {'tokenized_text': ['في',
   'عام',
   '1099',
   '،',
   'وقعت',
   'معركة',
   'حاسمة',
   'عند',
   'بوابات',
   'دمشق',
   '،',
   'والتي',
   'أ',
   'ُ',
   'طلق',
   'عليها',
   'اسم',
   'فتح',
   'القدس',
   '،',
   'حيث',
   'استطاع',
   'الصليبيون',
   'اختراق',
   'الحصون',
   'المحصنة',
   'للسيطرة',
   'على',
   'المدينة',
   'المقدسة',
   '.',
   'وفي',
   'ظل',
   'الاستعمار',
   '،',
   'لعبت',
   'جميلة',
   'بوحيرد',
   'دورا',
   'ً',
   'بارزا',
   'ً',
   'في',
   'النضال',
   'ضد',
   'المحتلين',
   '،',
   'حيث',
   'كانت',
   'رمزا',
  

In [16]:
num_steps = 500
batch_size =8
data_size = len(train_dataset)
num_batches = data_size // batch_size
num_epochs = max(1, num_steps // num_batches)

training_args = TrainingArguments(
    output_dir="models",
    learning_rate=5e-6,
    weight_decay=0.01,
    others_lr=1e-5,
    others_weight_decay=0.01,
    lr_scheduler_type="linear", #cosine
    warmup_ratio=0.1,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    focal_loss_alpha=0.75,
    focal_loss_gamma=2,
    num_train_epochs=num_epochs,
    evaluation_strategy="steps",
    save_steps = 100,
    save_total_limit=10,
    dataloader_num_workers = 0,
    use_cpu = False,
    report_to="none",
    )

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=model.data_processor.transformer_tokenizer,
    data_collator=data_collator,
)

trainer.train()

/home/ai/miniconda3/envs/nlp/lib/python3.11/site-packages/transformers/training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/tmp/ipykernel_846876/3557259569.py:28: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
../aten/src/ATen/native/cuda/Indexing.cu:1308: indexSelectLargeIndex: block: [352,0,0], thread: [32,0,0] Assertion `srcIndex < srcSelectDimSize` failed.
../aten/src/ATen/native/cuda/Indexing.cu:1308: indexSelectLargeIndex: block: [352,0,0], thread: [33,0,0] Assertion `srcIndex < srcSelectDimSize` failed.
../aten/src/ATen/native/cuda/Indexing.cu:1308: indexSelectLargeIndex: block: [352,0,0], thread: [34,0,0] Assertion `srcIndex < srcSelectDimSize` failed.
../aten/src/ATen/native/cuda/Indexing.cu:1308: indexSelectLargeIndex: block: [352,0,0], thread: [35,0

Skipping iteration due to error: CUDA error: device-side assert triggered
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.



RuntimeError: CUDA error: device-side assert triggered
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
# trained_model = GLiNER.from_pretrained("models/checkpoint-2209", load_tokenizer=True)


In [ ]:
texts = [
    """
    فاز ليونيل ميسي بجائزة الكرة الذهبية لعام 2023 بعد أداء مذهل مع نادي باريس سان جيرمان ومنتخب الأرجنتين في كأس العالم. يُعتبر ميسي من أفضل لاعبي كرة القدم في العالم، وقد قاد فريقه للفوز بلقب الدوري الفرنسي.
    """,
    """
    في عام 1945، انتهت الحرب العالمية الثانية بعد استسلام ألمانيا. وقّع الحلفاء اتفاقية في باريس، وأصبحت الأمم المتحدة رمزًا للسلام العالمي.
    """,
    
    """
    أعلنت شركة جوجل عن إطلاق منتج جديد في مؤتمرها السنوي في كاليفورنيا. المنتج الجديد، الذي طوره فريق بقيادة سوندار بيتشاي، يهدف إلى تحسين تجربة المستخدم.
    """,
    
    """
    نال الكاتب نجيب محفوظ جائزة نوبل للآداب عام 1988 عن روايته "أولاد حارتنا". تُرجم العمل إلى عدة لغات، وأُقيم احتفال كبير في القاهرة لتكريمه.
    """,
    
    """
    فازت السعودية باستضافة معرض إكسبو 2030 بعد منافسة قوية مع كوريا الجنوبية. سيُقام الحدث في الرياض، وسيشارك فيه عدد كبير من الشركات العالمية مثل أمازون ومايكروسوفت.
    """
]
labels = ["Person", "Award", "Organization", "Location", "Event"]

for i, text in enumerate(texts, 1):
    print(f"\nاختبار النص {i}:")
    entities = trained_model.predict_entities(text, labels, threshold=0.5)
    for entity in entities:
        print(entity["text"], "=>", entity["label"])